# Customer Purchase Segmentation using K-Means Clustering

This notebook segments mall customers into distinct groups based on their **annual income** and **spending score**, using the K-Means clustering algorithm.

**Workflow**
1. Load and explore the data
2. Clean and prepare the data
3. Visualize relationships between features
4. Find the optimal number of clusters (Elbow method + Silhouette score)
5. Fit the final K-Means model
6. Visualize and profile the resulting customer segments

**Dataset:** [Mall Customer Segmentation Data](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python) (`Mall_Customers.csv`). Place the CSV in the same folder as this notebook, or update the path below.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from kneed import KneeLocator
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
%matplotlib inline


## 1. Load the data

In [ ]:
DATA_PATH = 'Mall_Customers.csv'

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find '{DATA_PATH}'. Download the dataset from Kaggle "
        "(search 'Mall Customer Segmentation Data') and place it next to this notebook."
    )

df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Check for missing values
df.isnull().sum()


## 2. Clean and prepare the data

We don't need `CustomerID` for clustering, and we'll keep `Gender` aside for later profiling rather than dropping it entirely. Column names are simplified for convenience.


In [ ]:
df = df.rename(columns={
    'Annual Income (k$)': 'income',
    'Spending Score (1-100)': 'score',
    'Age': 'age',
    'Gender': 'gender',
    'CustomerID': 'customer_id'
})

df.head()


In [ ]:
df.shape


## 3. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df['age'], df['income'], alpha=0.7)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Annual Income (k$)')

axes[1].scatter(df['age'], df['score'], alpha=0.7, color='darkorange')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Spending Score (1-100)')

axes[2].scatter(df['income'], df['score'], alpha=0.7, color='seagreen')
axes[2].set_xlabel('Annual Income (k$)')
axes[2].set_ylabel('Spending Score (1-100)')

plt.tight_layout()
plt.show()


In [ ]:
sns.pairplot(df[['age', 'income', 'score']])
plt.savefig('pairplot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['income'], df['score'], color='black', alpha=0.7)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Annual Income vs. Spending Score')
plt.show()


The clearest separation between groups appears between **income** and **spending score**, so we'll cluster customers on these two features.


## 4. Finding the optimal number of clusters

### Elbow method

We fit K-Means for a range of cluster counts and look at the within-cluster sum of squares (WCSS / inertia). The "elbow" point indicates a good trade-off between cluster compactness and simplicity.


In [ ]:
X = df[['income', 'score']]

wcss = []
cluster_range = range(1, 11)

for k in cluster_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    wcss.append(km.inertia_)


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(list(cluster_range), wcss, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS (inertia)')
plt.title('Elbow Method')
plt.show()


In [ ]:
knee = KneeLocator(list(cluster_range), wcss, curve='convex', direction='decreasing')
optimal_k = knee.knee
print(f'Optimal number of clusters (elbow method): {optimal_k}')
knee.plot_knee()


### Silhouette score (cross-check)

The silhouette score measures how similar a point is to its own cluster vs. other clusters (closer to 1 is better). We use it to confirm the choice from the elbow method.


In [ ]:
silhouette_scores = []
sil_range = range(2, 11)

for k in sil_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    silhouette_scores.append(silhouette_score(X, labels))

plt.figure(figsize=(7, 5))
plt.plot(list(sil_range), silhouette_scores, marker='o', color='seagreen')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score')
plt.title('Silhouette Analysis')
plt.show()

best_k_silhouette = list(sil_range)[int(np.argmax(silhouette_scores))]
print(f'Optimal number of clusters (silhouette score): {best_k_silhouette}')


Both methods point to a similar value for `k`. We'll go with the elbow-method result.

## 5. Fit the final K-Means model

In [ ]:
N_CLUSTERS = optimal_k if optimal_k else best_k_silhouette

kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

centroids = kmeans.cluster_centers_
df.head()


## 6. Visualize the clusters

In [ ]:
plt.figure(figsize=(10, 6))

palette = sns.color_palette('tab10', n_colors=N_CLUSTERS)

for cluster_id in range(N_CLUSTERS):
    cluster_data = df[df['cluster'] == cluster_id]
    plt.scatter(
        cluster_data['income'], cluster_data['score'],
        color=palette[cluster_id], label=f'Cluster {cluster_id}', alpha=0.8
    )

plt.scatter(
    centroids[:, 0], centroids[:, 1],
    color='black', marker='X', s=200, label='Centroids'
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title(f'Customer Segments (k = {N_CLUSTERS})')
plt.legend()
plt.show()


## 7. Profile each segment

In [ ]:
cluster_profile = df.groupby('cluster')[['age', 'income', 'score']].mean().round(1)
cluster_profile['count'] = df.groupby('cluster').size()
cluster_profile


## 8. Conclusion

K-Means clustering on **annual income** and **spending score** reveals distinct customer segments, for example:

- High income, high spending — premium / target customers for loyalty programs
- High income, low spending — potential customers who need targeted marketing
- Low income, high spending — price-sensitive but engaged shoppers
- Low income, low spending — lowest priority segment
- Average income, average spending — the general customer base

These segments can guide targeted marketing campaigns, personalized offers, and customer retention strategies.

**Possible next steps:**
- Include `age` and `gender` in the clustering (with feature scaling, e.g. `StandardScaler`)
- Try other algorithms (DBSCAN, hierarchical clustering) for comparison
- Validate segments against real business outcomes (e.g. campaign response rates)
